# Ch. 5 · Assistant scientifique traçable

**IA pour les sciences — M1.** Recherche lexicale TF-IDF avec extraits vérifiables, sans LLM externe.

Exécutez les cellules dans l’ordre. La démarche et ses limites sont discutées dans le chapitre HTML correspondant. Les données artificielles utilisées, le cas échéant, servent uniquement à l’enseignement.

## Préparation de l’environnement

Dépendances usuelles : `numpy`, `pandas`, `matplotlib`, `scikit-learn` (et `scipy` pour le chapitre 6). Sur Colab, elles sont généralement disponibles.

### Étape 1

In [ ]:
documents = [
 {"id":"BIO-01", "source":"Note de laboratoire biologie", "texte":"Une classification doit être évaluée sur des échantillons indépendants."},
 {"id":"CHIM-02", "source":"Note de laboratoire chimie", "texte":"La solubilité dépend notamment de la température et du solvant."},
 {"id":"PHY-03", "source":"Note de laboratoire physique", "texte":"Les mesures répétées permettent d'estimer la dispersion expérimentale."},
 {"id":"GEO-04", "source":"Note de terrain géologie", "texte":"Une carte géologique dépend de l'échelle d'observation."}
]

### Étape 2

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

textes = [d["texte"] for d in documents]
vec = TfidfVectorizer(ngram_range=(1, 2),
                      strip_accents="unicode")
X = vec.fit_transform(textes)

def rechercher(question, k=2):
    q = vec.transform([question])
    scores = cosine_similarity(q, X).ravel()
    indices = scores.argsort()[::-1][:k]
    return [(documents[i], float(scores[i]))
            for i in indices if scores[i] > 0]

for doc, score in rechercher("évaluer une classification"):
    print(doc["id"], round(score, 3), doc["texte"])

### Étape 3

In [ ]:
def reponse_extractive(question):
    passages = rechercher(question, k=2)
    if not passages:
        return "Information absente du corpus."
    return "\n".join(
        f"[{doc['id']}] {doc['texte']}"
        for doc, score in passages)

print(reponse_extractive("comment évaluer une classification ?"))
print(reponse_extractive("quelle est la dose recommandée ?"))

### Étape 4

In [ ]:
# Exemple de contrôle sur une question connue
docs_retrouves = [d["id"] for d, s in
                 rechercher("mesures et dispersion", k=2)]
print("Documents récupérés :", docs_retrouves)

## Interprétation scientifique

1. Décrivez les données et leurs limites.
2. Justifiez la méthode et les métriques.
3. Discutez ce que les résultats permettent de conclure et ce qu’ils **ne permettent pas** de conclure.
4. Proposez au moins un contrôle de robustesse ou une expérience complémentaire.